# Sección 2 · Diversidad genética

*Ho, He, Fis, incertidumbre y la relación con la latitud en* Nothofagus pumilio *(lenga)*

> **Cómo usar este cuaderno.** Ejecuta las celdas en orden (Shift+Enter). Está armado en dos mitades: primero **nueve conceptos** —cada uno con su definición, un ejemplo numérico chico y lo que hay que retener— y después **siete bloques de análisis** que aplican esos conceptos a los datos del curso. Al final están los **ejercicios**, el **resumen** y la **Tarea**.
>
> Los datos son **simulados** (`00_simular_datos.R`): imitan rasgos generales de la lenga y sirven para practicar el flujo con una "verdad" conocida. No son datos biológicos reales.


In [ ]:
# ---- Preparación (ejecutar primero) ----
# Tamaño de gráficos en Jupyter
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 110)

source("R/helpers.R")
d <- load_clean(); G <- d$G; meta <- d$meta
meta$region <- assign_region(meta$lat)
pops <- aggregate(cbind(lat, lon, alt) ~ pop, data = meta, FUN = mean)

## Revisión de la tarea de la Sección 1

Antes de seguir, comparte tus 5 líneas de Métodos: qué umbrales elegiste y por qué.

- ¿Cuál te costó más justificar? Casi siempre es el **call rate**, porque parece técnico pero decide *quién* queda en el estudio.
- En la Sección 1 el umbral de **MAF** quedó fijo en 0.05 sin justificar. Hoy vas a medir cuánto cambia la diversidad según ese umbral, y con eso podrás escribirlo con argumento.
- Retomemos la hipótesis que anotaste: **¿esperas más diversidad genética al norte o al sur de la distribución? ¿Por qué?** Escríbela ahora. Al final de la sección la contrastamos.

**Datos con los que partimos:** `G` tiene **234 individuos × 1 174 SNP**, con un **3 % de genotipos faltantes**. Si en tu sesión aparece otra cosa, revisa la preparación.


## Objetivos de aprendizaje

Al terminar la sección deberías poder:

1. **Explicar con tus palabras** qué miden Ho, He y Fis, y por qué He y Fis responden preguntas distintas.
2. **Calcular** esas tres cantidades por población y **decidir** cuál reportar.
3. **Cuantificar la incertidumbre** de una estimación con bootstrap y leer un intervalo de confianza sin decir "significativo".
4. **Nombrar los sesgos** que mueven esos números (tamaño muestral, filtro de MAF, dependencia espacial) y decir en qué dirección empujan.


## Cómo está armada esta sección

| Parte | Qué hace | Celdas |
|---|---|---|
| **Conceptos 1–9** | La teoría mínima para no usar nada como caja negra | solo texto |
| **Análisis §2–§7** | Diversidad por población, gradiente, bootstrap, regiones, individuos, alelos privados | código |
| **Ejercicios E1–E4** | Donde se pone a prueba la interpretación | tu código |
| **Tarea** | El entregable que se revisa al inicio de la Sección 3 | tu trabajo |

**Vocabulario mínimo** (todo se define abajo, esta tabla es el mapa):

| Término | En una frase | Símbolo |
|---|---|---|
| Frecuencia alélica | Proporción de copias de un alelo en la muestra | $p$, $q$ |
| Frecuencia genotípica | Proporción de individuos con cada genotipo | $f(AA)$, $f(Aa)$ |
| Hardy-Weinberg (HW) | El modelo nulo: qué esperarías si nada especial pasa | $p^2+2pq+q^2=1$ |
| Ho | Heterocigosidad **observada**: cuántos heterocigotos hay | $H_o$ |
| He | Heterocigosidad **esperada** bajo HW a partir de $p$ | $H_e=2pq$ |
| Fis | Endogamia relativa: cuánto se aparta Ho de He | $F_{is}=1-H_o/H_e$ |
| Bootstrap | Remuestreo para estimar cuánto depende el resultado de la muestra | $B$ réplicas |
| MAF | Frecuencia del alelo **menos** común | $\min(p,q)$ |
| Alelo privado | Variante presente en una sola población o región | — |


## Concepto 1 · Frecuencias: en qué unidades se mide todo lo que viene

Toda la sección se construye sobre **dos** cantidades.

**Frecuencia genotípica.** De $n$ individuos, ¿qué proporción tiene cada genotipo? Si en una población de 12 árboles hay 6 heterocigotos, la frecuencia de heterocigotos es $6/12 = 0.5$. Es un **conteo**: no hay modelo, no hay supuestos.

**Frecuencia alélica.** ¿Qué proporción de las *copias* del gen es de cada alelo? Cada individuo diploide aporta dos copias, así que hay $2n$ copias. Si llamamos $p$ a la frecuencia del alelo alternativo (ALT) y $q = 1-p$ a la del de referencia (REF):

$$p = \frac{\text{número de copias ALT}}{2n} = \frac{\text{dosis media}}{2}$$

**Ejemplo numérico.** 5 individuos: dosis ALT `0, 1, 1, 2, 0`. Copias ALT: $0+1+1+2+0=4$ de $2 \times 5 = 10$. Entonces $p = 0.4$ y $q = 0.6$. Genotipos: dos homocigotos REF, dos heterocigotos, un homocigoto ALT.

> **Por qué importa el formato.** Nuestra matriz `G` guarda **dosis** (0/1/2/NA) porque con números se hace aritmética: la columna de un SNP promediada y dividida por 2 *es* su frecuencia alélica. Un error de orientación —transponer filas y columnas— no lanza ningún error: solo da números equivocados. Por eso la Sección 1 insistía en verificar la alineación con un `stopifnot`.

**Retén:** frecuencia genotípica = conteo; frecuencia alélica = conteo de copias. Todo lo demás de la sección sale de aquí.


## Concepto 2 · Hardy-Weinberg: el modelo nulo

Si en una población se cumplieran cinco condiciones —**apareamiento al azar, población grande, sin migración, sin mutación y sin selección**—, las frecuencias alélicas no cambiarían entre generaciones y los genotipos se repartirían así:

$$f(AA)=p^2 \qquad f(Aa)=2pq \qquad f(aa)=q^2 \qquad p^2+2pq+q^2=1$$

**Ejemplo numérico** (el del pizarrón): con $p=0.333$ y $q=0.667$,

| Genotipo | Frecuencia esperada | Con 12 individuos |
|---|---|---|
| Homocigoto REF ($p^2$) | $0.333^2 = 0.111$ | 1.3 |
| Heterocigoto ($2pq$) | $2(0.333)(0.667) = 0.444$ | 5.3 |
| Homocigoto ALT ($q^2$) | $0.667^2 = 0.444$ | 5.3 |

**Cómo se usa (y cómo no).** Hardy-Weinberg **no es una ley de la naturaleza**: es el *modelo nulo*, el punto de comparación. Cuando los datos se apartan de él, no está "mal" el modelo: hay algo que explicar —endogamia, subestructura, selección, errores de genotipado—. Y como es un modelo, **siempre** se aparta un poco: la pregunta nunca es "¿se cumple?", sino "¿se aparta más de lo que el azar explica?".

> **Ojo con el máximo.** $H_e = 2pq$ vale como máximo **0.5**, y solo si $p=q=0.5$. Si tus $H_e$ andan por 0.3, no es que la población esté "a la mitad de diversa": es que la mayoría de tus marcadores están lejos del 50-50. Los valores de diversidad no son porcentajes de una escala absoluta.


## Concepto 3 · Ho, He y Fis: variación y apareamiento son dos preguntas

**Ho (heterocigosidad observada).** Proporción de individuos heterocigotos, promediada sobre SNP. Se cuenta: si 6 de 12 son heterocigotos, $H_o = 0.500$.

**He (heterocigosidad esperada).** Lo que predice Hardy-Weinberg a partir de las frecuencias alélicas de esa población: $H_e = 2pq$. Con $p = 0.333$, $H_e = 0.444$.

**Fis (índice de endogamia).** Cuánto se aparta lo observado de lo esperado, en proporción:

$$F_{is} = 1 - \frac{H_o}{H_e}$$

Con los números de arriba: $1 - 0.500/0.444 = \mathbf{-0.126}$.

**Cómo se lee el signo:**

| $F_{is}$ | Qué significa | Causas típicas |
|---|---|---|
| $> 0$ (déficit) | Hay **menos** heterocigotos de los esperados | endogamia, subestructura (efecto Wahlund), alelos nulos o *dropout*, selección |
| $\approx 0$ | Lo observado coincide con lo esperado | apareamiento aproximadamente al azar |
| $< 0$ (exceso) | Hay **más** heterocigotos de los esperados | entrecruzamiento, apareamiento negativo por parentesco, n chico |

**Las dos ideas que no hay que soltar:**

1. **He mide variación; Fis mide apareamiento.** Una población puede tener mucha variación *y* estar endogámica. Reportar solo uno cuenta media historia.
2. **Un Fis positivo no diagnostica endogamia.** El mismo signo lo producen la subestructura y varios artefactos de genotipado. El número abre una pregunta; no la responde.


## Concepto 4 · La corrección de Nei: por qué $2n/(2n-1)$

Si estimas $p$ **con la misma muestra** con la que calculas $H_e$, el resultado baja un poco: los alelos raros suelen quedar subrepresentados por azar del muestreo, y $2\hat{p}(1-\hat{p})$ subestima el $2pq$ de la población. La corrección es un factor que depende solo del tamaño:

$$H_{e,\text{insesgada}} = 2\hat{p}(1-\hat{p}) \times \frac{2n}{2n-1}$$

| Individuos ($n$) | Copias ($2n$) | Factor | Con $H_e$ simple 0.300 |
|---|---|---|---|
| 4 | 8 | 1.143 | 0.343 |
| 6 | 12 | 1.091 | 0.327 |
| 11 | 22 | 1.048 | 0.314 |
| 12 | 24 | 1.043 | 0.313 |

**Consecuencia práctica:** dos poblaciones con distinto $n$ **no se comparan** en crudo, ni dos estudios con distinto $n$. Con 6 individuos el factor es 1.09: un 9 % de diferencia que no es biología.

> **Nota para más adelante:** esta corrección arregla el sesgo por $n$ chico, no el sesgo por **qué marcadores** tienes. Eso es otro problema (Concepto 7).


## Concepto 5 · Incertidumbre: qué es un intervalo de confianza y qué remuestrea un bootstrap

Un número solo no es un resultado: **cada estimación tiene un rango de valores plausibles**. El bootstrap lo estima de una forma muy simple:

1. Toma los SNP que tienes y **remuestréalos con reemplazo** (algunos entran dos veces, otros ninguna).
2. Recalcula la estadística (por ejemplo $H_e$) con esa muestra nueva.
3. Repite $B$ veces (acá, $B = 200$).
4. El rango entre el percentil 2.5 y el 97.5 de las réplicas es un **intervalo de confianza aproximado del 95 %**.

**Qué mide y qué no.** Mide cuánto depende tu estimación de **qué marcadores** te tocaron. **No** mide la incertidumbre por haber muestreado 12 árboles en vez de 50: eso es otro experimento y se responde con otro remuestreo (o se reconoce como limitación).

**Cómo se lee un solapamiento.** Si los intervalos de dos poblaciones **se solapan**, con esos datos la diferencia no está clara. Si **no** se solapan, hay una diferencia *consistente con estos marcadores* — que se reporta con magnitud e intervalo, no con la palabra "significativo".

> **Detalle honesto:** $B = 200$ es suficiente para ver el mecanismo en clase; en un trabajo real se usan $B = 1000$ o más, y se fija la semilla para que sea reproducible.


## Concepto 6 · Sesgo 1: el tamaño muestral

Con pocos individuos pasan **dos cosas a la vez**:

- $H_e$ se **subestima** (la corrección de Nei lo compensa solo en parte).
- La **dispersión entre poblaciones aumenta**: cada estimación es más ruidosa.

Por eso un gráfico de "He promedio versus $n$" no es un detalle metodológico: es la forma de **elegir tu $n$ mínimo aceptable** en un estudio real, en vez de heredarlo de un paper. Es el ejercicio **E2**.

**Cómo saber si está bien tu ejercicio:** si la curva se aplana (la media se estabiliza y la dispersión se angosta) alrededor de $n = 8\text{–}12$, ese es tu umbral; si sigue moviéndose al llegar a 12, tu diseño necesita más individuos.


## Concepto 7 · Sesgo 2: MAF y sesgo de verificación

**MAF** es la frecuencia del alelo menos común: $\mathrm{MAF}=\min(p,q)$, entre 0 y 0.5. Como $H_e = 2pq$ es **creciente** en $p$ hasta 0.5, subir el umbral de MAF **saca los SNP de menor $H_e$**… y por lo tanto **la $H_e$ promedio sube**. No cambió la población: cambió el filtro.

Con nuestros datos lo vas a ver en el ejercicio **E3**:

| Filtro | SNP que quedan | $H_e$ promedio |
|---|---|---|
| MAF > 0.05 (el de la Sección 1) | 1 174 | 0.325 |
| MAF > 0.10 | 1 017 | 0.356 |
| MAF > 0.20 | 725 | 0.405 |

**Sesgo de verificación (*ascertainment bias*).** Un panel de SNP no es una muestra al azar del genoma: los marcadores se descubren y se eligen con algún criterio (secuenciar pocos individuos, diseñar un arreglo, exigir calidad). Si ese criterio favorece variantes comunes, la diversidad queda **subestimada** por construcción, y las comparaciones entre poblaciones heredan el sesgo. Es la razón por la que "comparar con la literatura" exige el mismo tipo de marcador **y** el mismo filtro.

**Retén:** la diversidad que reportas no es una propiedad absoluta de la población; es una propiedad de **la población + tu filtro**.


## Concepto 8 · Riqueza alélica y alelos privados: dependen de $n$

**Riqueza alélica** es cuántos alelos distintos hay en una población. **Alelo privado** es una variante presente en una población o región y **ausente** en las demás: sugiere aislamiento, refugios o historia independiente.

El problema: ambos **crecen con el número de individuos examinados**, porque más individuos = más variantes raras detectadas. Comparar una región con 58 individuos contra otra con 94 sin corregir es comparar esfuerzo de muestreo, no biología. La solución estándar es la **rarefacción**: recalcular la riqueza al $n$ más pequeño del conjunto (o usar un estimador que lo haga).

> En esta sección el resultado de alelos privados va a dar **0 en las tres regiones**. No es un error del código: la estructura simulada es un **gradiente continuo**, sin regiones aisladas, y un gradiente no produce variantes exclusivas. La lección queda igual: antes de interpretar un número de alelos privados, mira los $n$.


## Concepto 9 · Geografía: las poblaciones no son observaciones independientes

Un modelo como `lm(He ~ lat)` supone que los residuos son **independientes**. Pero 20 poblaciones a lo largo de 2 000 km no lo son: las vecinas comparten historia, flujo génico y refugios. Eso se llama **pseudorreplicación** —el mismo problema que contar células como si fueran donantes—.

Consecuencia concreta: **los $p$-valores ingenuos son demasiado optimistas**. Un $p = 10^{-5}$ sobre 20 poblaciones correlacionadas no equivale a la misma evidencia que un $p = 10^{-5}$ sobre 20 sitios independientes.

Qué hacer, en orden de esfuerzo:

1. **Escribirlo** en Limitaciones y hablar de "evidencia sugerente".
2. **Permutar** de forma restringida (dentro de regiones) en vez de al azar.
3. **Modelar** la dependencia espacial: `nlme::gls` con `corExp`, o un enfoque de aislamiento por distancia (Sección 3).

Es el ejercicio **E4**. La idea clave: con dependencia espacial, el $p$-valor ingenuo no es un resultado, es un punto de partida.


## Análisis §2 · Diversidad por población

`diversity_by_pop()` devuelve, por población: $n$, $H_o$, $H_e$ (con la corrección de Nei), $F_{is}$ y la proporción de SNP polimórficos. La tabla queda ordenada de norte a sur.

**¿Qué esperas?** Anótalo antes de mirar: ¿en qué rango crees que va a estar $H_e$ (recuerda el máximo 0.5)? ¿Y $F_{is}$ positivo o negativo?


In [ ]:
div <- diversity_by_pop(G, meta$pop)
div <- merge(div, pops, by = "pop")
div <- div[order(div$lat, decreasing = TRUE), ]   # de norte a sur
round_df <- function(x, k = 3) { x[sapply(x, is.numeric)] <- round(x[sapply(x, is.numeric)], k); x }
head(round_df(div), 8)

**Cómo leer la tabla.** Cuatro cosas, en este orden:

1. **$H_e$ entre 0.318 y 0.337**: un rango estrecho. El máximo está en **L01** (la más al norte) y el mínimo en **L17**.
2. **$H_e \approx 0.33$, no 0.5**: es lo esperable tras un filtro de MAF; el máximo teórico exige $p=q=0.5$.
3. **$F_{is}$ positivo en las veinte poblaciones** (0.017–0.045): déficit leve y sistemático. ¿Endogamia? ¿Subestructura? ¿Artefacto? Es una pregunta abierta (Concepto 3), no una respuesta.
4. **`prop_polim` entre 0.87 y 0.93**: hay SNP que en el conjunto son variables pero que **dentro de una población** están fijos. Polimorfismo, otra vez, es relativo a la muestra.

Y mira `n_ind`: **10 a 12**. No es parejo, y eso importa en el bloque siguiente.


## Análisis §3 · ¿Hay un gradiente con la latitud?

Primero el gráfico, después los números: cada punto es una población; la línea es un ajuste lineal y la banda, su incertidumbre.

**¿Qué esperas?** Si la diversidad baja hacia el sur, la pendiente debería ser… *(piénsalo: la latitud va en negativos, así que "más al norte" es "menos negativo")*.


In [ ]:
ggplot(div, aes(lat, He)) + geom_point(size = 3) + geom_smooth(method = "lm", se = TRUE) +
  labs(x = "Latitud (°)", y = "He") 

Ahora los dos números que sostienen (o no) lo que ves: el ajuste lineal y una correlación de Spearman, que no supone linealidad.


In [ ]:
m <- lm(He ~ lat, data = div)
summary(m)$coefficients
cor.test(div$lat, div$He, method = "spearman")

**Qué dicen.** Pendiente $+0.00077$ por grado de latitud, $R^2 = 0.67$ (dos tercios de la variación en $H_e$ se asocian a latitud) y $\rho = 0.82$ con $p \approx 10^{-5}$.

**Y qué no dicen.** Dos advertencias antes de escribir la conclusión:

- **$n = 20$ poblaciones no independientes** (Concepto 9): la evidencia es *sugerente*, no definitiva.
- **El rango es angosto**: de 0.318 a 0.337 es un 6 % de diferencia entre extremos. Es un **gradiente suave**, y así se reporta.

Una frase que sí se puede escribir: *"observamos una tendencia a menor diversidad hacia el sur (pendiente, $R^2$, IC), consistente con un gradiente suave, en un diseño de 20 poblaciones con dependencia espacial"*.


## Análisis §4 · Incertidumbre: bootstrap sobre SNP

Compara dos poblaciones de los extremos (L01, norte; L20, sur) remuestreando **SNP con reemplazo**, $B = 200$ réplicas, con semilla fija.

**¿Qué esperas?** ¿Los intervalos van a solaparse o no? ¿Qué significa cada caso?


In [ ]:
set.seed(1)
boot_he <- function(Gk, B = 200) {
  replicate(B, {
    cols <- sample(ncol(Gk), replace = TRUE)
    diversity_by_pop(Gk[, cols], rep("x", nrow(Gk)))$He
  })
}
# ejemplo para una población (L01) y otra (L20)
b1  <- boot_he(G[meta$pop == "L01", ]); b20 <- boot_he(G[meta$pop == "L20", ])
rbind(L01 = quantile(b1, c(.025, .5, .975)), L20 = quantile(b20, c(.025, .5, .975)))

**Cómo leerlo.** L01: $H_e = 0.337$, IC 95 % **[0.328; 0.347]**. L20: $H_e = 0.320$, IC **[0.311; 0.329]**. Los intervalos **apenas se tocan**: la diferencia entre extremos es consistente con estos marcadores, pero no enorme.

Y el recordatorio del Concepto 5: este intervalo **no** incorpora la incertidumbre de haber muestreado 12 árboles. Si te preguntas por eso, la respuesta no es el bootstrap: es el ejercicio E2.


## Análisis §5 · Comparación entre regiones

Se agrupa por latitud (Norte, Centro, Sur) y se compara con **Kruskal-Wallis**, un test no paramétrico pensado para pocos grupos.

**¿Qué esperas?** ¿Qué región tendrá más dispersión? Y la pregunta incómoda: ¿y si el esfuerzo de muestreo no es el mismo?


In [ ]:
div$region <- assign_region(div$lat)
aggregate(cbind(Ho, He, Fis) ~ region, data = div, FUN = mean)
kruskal.test(He ~ region, data = div)
ggplot(div, aes(region, He, fill = region)) + geom_boxplot(alpha = .6, show.legend = FALSE) +
  geom_jitter(width = .1)

**Qué sale.** $H_e$ media: Norte **0.330**, Centro **0.326**, Sur **0.320**. Kruskal-Wallis $p = 0.0021$. Mismo orden que la recta, misma magnitud pequeña.

**Y el dato que manda sobre la interpretación:** los tamaños **no son parejos** — Norte 5 poblaciones y 58 individuos, Centro 8 y 94, Sur 7 y 82. Con el centro con casi el doble de individuos que el norte, **biología y esfuerzo de muestreo están confundidos**. La comparación honesta exige igualar $n$ (rarefacción o submuestreo, ejercicio E2), y decirlo en Limitaciones.


## Análisis §6 · Diversidad individual

Bajamos un nivel: la **heterocigosidad multilocus** de cada individuo (proporción de sus SNP que son heterocigotos). Sirve para ver variación **dentro** de cada población y detectar individuos atípicos.

**¿Qué esperas?** ¿Habrá individuos fuera de rango? ¿Qué podría explicarlos?


In [ ]:
het_ind <- rowMeans(G == 1, na.rm = TRUE)       # heterocigosidad multilocus
ind_df  <- data.frame(meta, het = het_ind)
ggplot(ind_df, aes(reorder(pop, -lat), het)) + geom_boxplot() +
  labs(x = "Población (norte → sur)", y = "Heterocigosidad individual") +
  theme(axis.text.x = element_text(angle = 90, vjust = .5))

**Qué buscar.** Media 0.316, con **cuatro individuos atípicos**: tres por arriba (L08_11, 0.369 el más alto) y uno por abajo (**L20_02, 0.275**).

Un individuo así puede ser un migrante, una muestra mal etiquetada, contaminación, o simplemente azar de marcadores. **No se concluye acá**: se anota y se vuelve a él cuando toquemos estructura. En datasets reales este gráfico es el primer detector de problemas de etiquetado.


## Análisis §7 · Alelos privados por región

Un **alelo privado** está en una región y ausente en las otras dos. Se calcula sobre frecuencias por región y se prueba en ambos sentidos (ALT presente solo aquí; REF presente solo aquí).

**¿Qué esperas?** Con estructura en gradiente, ¿cuántos alelos exclusivos crees que habrá?


In [ ]:
reg <- factor(meta$region)
Pr  <- t(sapply(levels(reg), function(r) colMeans(G[reg == r, ], na.rm = TRUE) / 2))
privados <- sapply(seq_len(nrow(Pr)), function(i) {
  otros <- Pr[-i, , drop = FALSE]
  sum(Pr[i, ] > 0 & colSums(otros > 0) == 0) +      # ALT presente solo aquí
  sum(Pr[i, ] < 1 & colSums(otros < 1) == 0)        # REF presente solo aquí
})
setNames(privados, levels(reg))
table(reg)     # ¡el esfuerzo de muestreo difiere!

**Resultado: 0, 0 y 0.** Y no es un bug: la estructura simulada es un **gradiente continuo**, sin regiones aisladas, y un gradiente sin barreras no produce variantes exclusivas (Concepto 8).

Fíjate además en `table(reg)`: **58, 94 y 82 individuos**. Si el resultado hubiera sido, digamos, 20, 8 y 3, **no** podríamos decir nada sin rarefaccionar primero: el número de alelos privados sigue al tamaño de muestra.


---

## Ejercicios

Los cuatro ejercicios apuntan a lo mismo: **que la interpretación no dependa del número que salió primero**. Cada uno trae qué hacer, una pista y cómo saber si está bien.


**E1 · Tres métricas, tres conclusiones posibles.** Ajusta `lm(He ~ lat)`, `lm(Ho ~ lat)` y `lm(Fis ~ lat)`. Grafica los tres y compara pendientes y $R^2$.

- **Pista:** parte de `div` y usa `lm()` + `coef(summary(...))`; para los tres gráficos, `patchwork` o tres gráficos seguidos.
- **Cómo saber si está bien:** puedes nombrar una métrica que muestre gradiente y otra que no. Si las tres coinciden, sospecha del filtro antes que de la biología.
- **Idea clave:** se reportan las tres. Elegir la que "salió mejor" es sesgo de confirmación.


In [ ]:
# TU CÓDIGO AQUÍ

**E2 · Sesgo por tamaño de muestra.** Submuestrea 4, 6, 8 y 12 individuos por población (20 réplicas cada uno), recalcula $H_e$ y $F_{is}$, y grafica media y dispersión versus $n$.

- **Pista:** `split(seq_len(nrow(G)), meta[["pop"]])` te da los índices por población; con `sample()` eliges $n$ de cada una.
- **Cómo saber si está bien:** la dispersión se angosta al crecer $n$, y la media de $H_e$ **sube** un poco (corrección de Nei + menos sesgo).
- **Idea clave:** con $n$ chico la dispersión es alta. Define el $n$ mínimo aceptable **mirando esa curva**, no copiándolo.


In [ ]:
# Pista: split(seq_len(nrow(G)), meta$pop) + sample()

**E3 · Sesgo de verificación (MAF).** Recalcula la diversidad con MAF > 0.05, 0.10 y 0.20 y compara la relación $H_e \sim$ latitud.

- **Pista:** la frecuencia alélica global de cada SNP es `colMeans(G, na.rm = TRUE)/2`; el MAF es `pmin(p, 1-p)`; filtra columnas con `G[, keep]`.
- **Cómo saber si está bien:** al subir el umbral quedan menos SNP, la $H_e$ media **sube**, y la *dirección* del gradiente se mantiene (los números están en el Concepto 7 para que los compares).
- **Idea clave:** el filtro mueve el nivel, no la biología — pero un lector solo puede comparar tu $H_e$ con otro estudio si el filtro coincide.


In [ ]:
# TU CÓDIGO AQUÍ

**E4 · Dependencia espacial.** Reemplaza la regresión por una versión que considere la autocorrelación espacial (`nlme::gls` con `corExp` sobre lat/lon, o una prueba de permutación que permute poblaciones **dentro** de regiones). Discute qué cambia.

- **Pista:** guarda los coeficientes y el $p$ del modelo ingenuo antes de cambiarlo; la comparación es el resultado.
- **Cómo saber si está bien:** el $p$ se vuelve menos optimista (o el intervalo se ensancha) y puedes explicar por qué.
- **Idea clave:** con dependencia espacial, los $p$-valores ingenuos sobreestiman la evidencia; el resultado es la diferencia entre ambos análisis.


In [ ]:
# TU CÓDIGO AQUÍ

## Errores y dudas frecuentes

- **"Mi bootstrap da intervalos muy anchos."** Es lo esperable con pocos individuos y pocos marcadores. No es un error; es información sobre tu diseño.
- **"$F_{is}$ me da negativo en una población."** Posible con $n$ pequeño. No interpretes un valor aislado: mira el conjunto y su intervalo.
- **"¿Por qué la $H_e$ insesgada?"** Porque estimar $p$ con la misma muestra subestima $2pq$; el factor $2n/(2n-1)$ compensa (Concepto 4).
- **"¿Puedo comparar mi $H_e$ con la literatura?"** Solo si coinciden **tipo de marcador**, **filtro de MAF**, **corrección** y, en lo posible, $n$.
- **"$F_{is}$ positivo ⇒ endogamia."** No: también lo producen subestructura (Wahlund), alelos nulos y *dropout*. El número abre la pregunta.
- **"$p < 0.05$ ⇒ hay gradiente."** Con poblaciones correlacionadas, no. Mira el Concepto 9 y el E4.
- **"$H_e$ bajó al filtrar más."** Al contrario: si subes el umbral de MAF, $H_e$ **sube**. Si bajó, revisa si filtraste por call rate o cambiaste de conjunto de SNP.


## Resumen: las ideas que te llevas

1. **Dos preguntas, dos números.** $H_e$ describe la **variación**; $F_{is}$ describe el **apareamiento**. Ninguno reemplaza al otro.
2. **Hardy-Weinberg es el modelo nulo**, no una ley: apartarse de él es el dato, no el fallo.
3. **Un número sin incertidumbre no es un resultado.** El bootstrap cuantifica una parte de la incertidumbre; el tamaño muestral y el filtro de MAF son sesgos que hay que **nombrar**.
4. **La independencia espacial casi siempre se viola.** Con poblaciones correlacionadas, un $p$-valor ingenuo es optimista.
5. **La diversidad que reportas depende del filtro y del $n$**: es una propiedad del dato *y* del diseño.

> *Vuelve a tu hipótesis del inicio.* ¿Coincide con lo observado (tendencia a menor diversidad hacia el sur, gradiente suave)? ¿Qué parte de esa señal podría ser diseño de muestreo?


## Tarea (se revisa al comienzo de la Sección 3)

1. **Gráfico publicable** de $H_e$ por población, ordenado por latitud, con **barras de error bootstrap**. Ejes rotulados, unidades, y una nota al pie con $B$ y la semilla.
2. **Una frase de conclusión sin la palabra "significativo"**: magnitud + dirección + incertidumbre. Ejemplo: *"observamos una tendencia a menor diversidad hacia el sur (pendiente $x$, IC 95 % $[a; b]$), con incertidumbre y dependencia espacial entre poblaciones"*.
3. **Tres líneas de Limitaciones** que nombren, como mínimo: dependencia espacial, desigualdad de $n$ entre regiones, y el efecto del filtro de MAF sobre $H_e$.

*Pregunta de reflexión (para discutir al inicio de la Sección 3):* si observaras menor diversidad al sur, ¿qué hipótesis alternativas lo explicarían —efectos fundadores post-glaciares, menor tamaño poblacional, refugios, error de muestreo, sesgo de verificación— y **qué dato adicional** distinguiría una de otra?
